# 1. Thermal-IR Drone Detection Pipeline

**AAI-590 Capstone | Stage 2 — dataset preparation**

This notebook is the reproducible entry point for the thermal-IR workflow. It validates/reuses the prepared data, trains the configured baseline, selects an image-level decision threshold on validation data, and evaluates the isolated test split.

## 1.1 Deployment question and model contract

The system answers one operational question: **Is there a drone in this image?** This is implemented as a one-class object detector, not as a drone-versus-bird classifier. The detector outputs zero or more `drone` boxes; the image-level answer is **drone present** when at least one box exceeds the validation-selected confidence threshold, otherwise **no drone present**.

| Source content | Export role | YOLO label behavior |
| --- | --- | --- |
| Valid Dataset 1 drone frame or Dataset 2 `DRONE` frame | Drone-positive example | One class-0 (`drone`) bounding box |
| Dataset 2 `BIRD` frame | Hard-negative no-drone example | Empty label file; the source bird box is retained only for audit/QA |
| Invalid Dataset 1 drone annotation | Excluded pending review | No image/label export; never silently made a negative |

- The only model class is `0: drone`.
- Bird recordings are labelled as no-drone hard negatives: their bird boxes are **not** exported to YOLO.
- Source images and videos remain immutable.
- Splits must be recording/sequence-disjoint.
- Dataset 1 supplied test folders are inference-only and are excluded from metrics.

## 1.2 Dataset sources and current audit status

| Source | Intended use | Local raw destination | Status |
| --- | --- | --- | --- |
| Official 4th Anti-UAV Challenge / Zenodo 15103888 | Additional thermal drone examples | `data/thermal-ir/raw/anti_uav_dataset1/` | Dataset 1 `train` is frame-labelled; supplied test folders are inference-only |
| DroneDetectionThesis/Drone-detection-dataset | Thermal drone positives and bird hard negatives | `data/thermal-ir/raw/drone_detection_dataset2/` | `.mat` labels require `mcos-decoder`; video/label alignment needs validation |

Read the complete data and pipeline reference before processing: [`docs/thermal_ir_pipeline.md`](../docs/thermal_ir_pipeline.md).

Dataset 2 is present locally. Dataset 1 is being copied manually; do not execute Dataset 1 manifest, conversion, or split cells until its raw-file gate reports complete.

In [1]:
from __future__ import annotations

from pathlib import Path
import sys

# Support running from either the repository root or notebooks/.
WORKING_DIRECTORY = Path.cwd().resolve()
if (WORKING_DIRECTORY / '.git').is_dir():
    REPO_ROOT = WORKING_DIRECTORY
elif (WORKING_DIRECTORY.parent / '.git').is_dir():
    REPO_ROOT = WORKING_DIRECTORY.parent
else:
    raise RuntimeError('Open this notebook from the AAI-590 repository or its notebooks/ directory.')

THERMAL_ROOT = REPO_ROOT / 'data' / 'thermal-ir'
RAW_ROOT = THERMAL_ROOT / 'raw'
DATASET1_ROOT = RAW_ROOT / 'anti_uav_dataset1'
DATASET2_ROOT = RAW_ROOT / 'drone_detection_dataset2' / 'Data' / 'Video_IR'
MANIFEST_ROOT = THERMAL_ROOT / 'manifests'
QA_ROOT = THERMAL_ROOT / 'interim' / 'qa_overlays'
YOLO_ROOT = THERMAL_ROOT / 'processed' / 'yolo_v1'
RESULTS_ROOT = REPO_ROOT / 'results' / 'thermal-ir'

# YOLO has one target class. A bird must never appear in a label file.
CLASS_NAMES = {0: 'drone'}
RANDOM_SEED = 590

print(f'Python: {sys.version.split()[0]}')
print(f'Repository: {REPO_ROOT}')
print(f'Thermal data root: {THERMAL_ROOT}')

Python: 3.11.7
Repository: D:\Capstone\aai-590-capstone
Thermal data root: D:\Capstone\aai-590-capstone\data\thermal-ir


## 1.3 Environment setup

Create a dedicated environment from the repository root before running processing cells:

```powershell
uv venv .venv
.\.venv\Scripts\Activate.ps1
uv pip install pandas pillow opencv-python-headless mcos-decoder pyyaml matplotlib seaborn
```

`ultralytics` is intentionally omitted at this stage. Add it only after the prepared dataset and training configuration are approved.

In [2]:
from importlib.util import find_spec

REQUIRED_PACKAGES = {
    'pandas': 'pandas',
    'Pillow': 'PIL',
    'OpenCV': 'cv2',
    'mcos-decoder': 'mcos_decoder',
    'PyYAML': 'yaml',
    'matplotlib': 'matplotlib',
    'seaborn': 'seaborn',
}
missing_packages = [name for name, module in REQUIRED_PACKAGES.items() if find_spec(module) is None]
if missing_packages:
    print('Missing packages:', ', '.join(missing_packages))
    print('Install them using the previous cell before running data-processing cells.')
else:
    print('All Stage 2 processing dependencies are available.')

All Stage 2 processing dependencies are available.


In [3]:
# These expected counts were established in the read-only source-data audit.
EXPECTED_RAW_FILE_COUNTS = {
    'Dataset 1 Anti-UAV': 763_717,
    'Dataset 2 Drone Detection': 736,
}
RAW_DATASET_ROOTS = {
    'Dataset 1 Anti-UAV': DATASET1_ROOT,
    'Dataset 2 Drone Detection': RAW_ROOT / 'drone_detection_dataset2',
}

def count_files(path: Path) -> int:
    """Count dataset payload files, excluding copied Git metadata."""
    return sum(
        1
        for candidate in path.rglob('*')
        if candidate.is_file() and '.git' not in candidate.relative_to(path).parts
    )

def raw_copy_status() -> dict[str, dict[str, object]]:
    report = {}
    for name, path in RAW_DATASET_ROOTS.items():
        actual = count_files(path) if path.is_dir() else 0
        expected = EXPECTED_RAW_FILE_COUNTS[name]
        report[name] = {
            'path': path,
            'actual_files': actual,
            'expected_files': expected,
            'complete': actual == expected,
        }
    return report

RAW_STATUS = raw_copy_status()
for name, status in RAW_STATUS.items():
    label = 'COMPLETE' if status['complete'] else 'INCOMPLETE'
    print(f"{label:>10}  {name}: {status['actual_files']:,}/{status['expected_files']:,} payload files")

def require_complete_raw_data() -> None:
    incomplete = [name for name, status in RAW_STATUS.items() if not status['complete']]
    if incomplete:
        raise RuntimeError(
            'Raw copy is incomplete for: ' + ', '.join(incomplete) +
            '. Complete the manual copy, re-run the raw-completeness cell, then continue.'
        )

  COMPLETE  Dataset 1 Anti-UAV: 763,717/763,717 payload files
  COMPLETE  Dataset 2 Drone Detection: 736/736 payload files


In [4]:
# Read-only preflight: this cell deliberately does not create, copy, or modify data.
for name, path in {
    'Dataset 1 Anti-UAV train': DATASET1_ROOT / 'train',
    'Dataset 2 Video_IR': DATASET2_ROOT,
}.items():
    state = 'READY' if path.is_dir() else 'NOT YET POPULATED'
    print(f'{state:>18}  {name}: {path}')

print('\nThe raw-completeness gate must pass before manifest generation or extraction.')

             READY  Dataset 1 Anti-UAV train: D:\Capstone\aai-590-capstone\data\thermal-ir\raw\anti_uav_dataset1\train
             READY  Dataset 2 Video_IR: D:\Capstone\aai-590-capstone\data\thermal-ir\raw\drone_detection_dataset2\Data\Video_IR

The raw-completeness gate must pass before manifest generation or extraction.


# 2. Source manifests and annotation validation

Run the following cells only after the raw-copy gate is complete. They read immutable source files and write only derived CSV manifests under `data/thermal-ir/manifests/`. No source image, video, JSON, or MATLAB file is modified.

The issue log is intentionally comprehensive: invalid boxes, missing paired files, array-length mismatches, and Dataset 2 annotation/video frame-count mismatches are all retained rather than silently corrected.

In [5]:
import json
from collections import Counter

import cv2
import pandas as pd
from mcos_decoder import load_groundtruth
from PIL import Image, UnidentifiedImageError

IMAGE_SUFFIXES = {'.jpg', '.jpeg', '.png'}
# Dataset 2 policy: drone boxes are positives; bird frames are explicit no-drone examples.
DATASET2_EXCLUDED_CLASSES = {'AIRPLANE', 'HELICOPTER'}
DATASET2_EXPORT_POLICY = {'DRONE': 'positive_drone', 'BIRD': 'negative_bird'}

def source_frames(sequence_dir: Path) -> list[Path]:
    """Return deterministic, frame-number ordered source images for one Dataset 1 sequence."""
    return sorted(
        (path for path in sequence_dir.iterdir() if path.is_file() and path.suffix.lower() in IMAGE_SUFFIXES),
        key=lambda path: path.name,
    )

def xywh_issue(box, width: int, height: int) -> str | None:
    """Validate a pixel-space top-left xywh box without changing it."""
    if not isinstance(box, (list, tuple)) or len(box) != 4:
        return 'invalid_box_shape'
    try:
        x, y, box_width, box_height = (float(value) for value in box)
    except (TypeError, ValueError):
        return 'non_numeric_box'
    if not all(pd.notna(value) for value in (x, y, box_width, box_height)):
        return 'non_finite_box'
    if box_width <= 0 or box_height <= 0:
        return 'non_positive_box_dimensions'
    if x < 0 or y < 0:
        return 'negative_box_origin'
    if x + box_width > width or y + box_height > height:
        return 'box_exceeds_image_boundary'
    return None

def manifest_issue(source_dataset: str, source_sequence: str, issue_type: str, **details) -> dict:
    return {
        'source_dataset': source_dataset,
        'source_sequence': source_sequence,
        'issue_type': issue_type,
        **details,
    }

In [6]:
def inspect_dataset1_train() -> tuple[pd.DataFrame, pd.DataFrame]:
    """Inspect Dataset 1 `train` only; do not alter its frames or annotations."""
    sequence_rows, issues = [], []
    train_root = DATASET1_ROOT / 'train'
    for sequence_dir in sorted(path for path in train_root.iterdir() if path.is_dir()):
        sequence_id = sequence_dir.name
        labels_path = sequence_dir / 'IR_label.json'
        frames = source_frames(sequence_dir)
        row = {
            'source_dataset': 'anti_uav_dataset1',
            'source_partition': 'train',
            'source_sequence': sequence_id,
            'frame_count': len(frames),
            'annotation_path': labels_path.relative_to(REPO_ROOT).as_posix(),
        }
        if not frames:
            issues.append(manifest_issue('anti_uav_dataset1', sequence_id, 'missing_source_frames'))
            sequence_rows.append(row)
            continue
        with Image.open(frames[0]) as first_frame:
            width, height = first_frame.size
        row.update({'image_width': width, 'image_height': height})
        if not labels_path.is_file():
            issues.append(manifest_issue('anti_uav_dataset1', sequence_id, 'missing_annotation_file'))
            sequence_rows.append(row)
            continue
        try:
            labels = json.loads(labels_path.read_text(encoding='utf-8'))
            exists, boxes = labels.get('exist'), labels.get('gt_rect')
            if not isinstance(exists, list) or not isinstance(boxes, list):
                raise ValueError('missing `exist` or `gt_rect` list')
        except (OSError, ValueError, json.JSONDecodeError) as error:
            issues.append(manifest_issue('anti_uav_dataset1', sequence_id, 'unreadable_annotation', detail=str(error)))
            sequence_rows.append(row)
            continue
        row.update({
            'exist_entries': len(exists),
            'box_entries': len(boxes),
            'visible_entries': sum(value == 1 for value in exists),
            'absent_entries': sum(value != 1 for value in exists),
            'annotation_keys': ','.join(sorted(labels)),
        })
        for frame_index, frame_path in enumerate(frames, start=1):
            if frame_path.stat().st_size == 0:
                issues.append(manifest_issue(
                    'anti_uav_dataset1', sequence_id, 'unreadable_source_frame',
                    original_frame=frame_index, detail='zero-byte image file',
                ))
        if len(frames) != len(exists) or len(exists) != len(boxes):
            issues.append(manifest_issue(
                'anti_uav_dataset1', sequence_id, 'frame_annotation_length_mismatch',
                frame_count=len(frames), exist_entries=len(exists), box_entries=len(boxes),
            ))
        for frame_index, (exists_flag, box) in enumerate(zip(exists, boxes), start=1):
            if exists_flag == 1:
                issue_type = xywh_issue(box, width, height)
                if issue_type:
                    issues.append(manifest_issue(
                        'anti_uav_dataset1', sequence_id, issue_type,
                        original_frame=frame_index, bbox=json.dumps(box),
                    ))
        sequence_rows.append(row)
    return pd.DataFrame(sequence_rows), pd.DataFrame(issues)

In [7]:
def inspect_dataset2_videos() -> tuple[pd.DataFrame, pd.DataFrame]:
    """Inspect all Dataset 2 IR pairs and retain excluded-class inventory information."""
    video_rows, issues = [], []
    for labels_path in sorted(DATASET2_ROOT.glob('IR_*_LABELS.mat')):
        parts = labels_path.stem.split('_')
        source_class = parts[1] if len(parts) >= 3 else 'UNKNOWN'
        recording_id = labels_path.name.removesuffix('_LABELS.mat')
        video_path = labels_path.with_name(f'{recording_id}.mp4')
        row = {
            'source_dataset': 'drone_detection_dataset2',
            'source_sequence': recording_id,
            'source_class': source_class,
            'export_policy': DATASET2_EXPORT_POLICY.get(source_class, 'excluded'),
            'eligible_for_single_class_detector': source_class in DATASET2_EXPORT_POLICY,
            'video_path': video_path.relative_to(REPO_ROOT).as_posix(),
            'annotation_path': labels_path.relative_to(REPO_ROOT).as_posix(),
        }
        if not video_path.is_file():
            issues.append(manifest_issue('drone_detection_dataset2', recording_id, 'missing_video_file'))
            video_rows.append(row)
            continue
        capture = cv2.VideoCapture(str(video_path))
        try:
            frame_count = int(capture.get(cv2.CAP_PROP_FRAME_COUNT))
            width = int(capture.get(cv2.CAP_PROP_FRAME_WIDTH))
            height = int(capture.get(cv2.CAP_PROP_FRAME_HEIGHT))
            fps = float(capture.get(cv2.CAP_PROP_FPS))
        finally:
            capture.release()
        boxes = load_groundtruth(labels_path)
        visible_boxes = [box for box in boxes if box is not None]
        row.update({
            'video_frame_count': frame_count,
            'annotation_entries': len(boxes),
            'visible_entries': len(visible_boxes),
            'absent_entries': len(boxes) - len(visible_boxes),
            'image_width': width,
            'image_height': height,
            'fps': fps,
        })
        if frame_count != len(boxes):
            issues.append(manifest_issue(
                'drone_detection_dataset2', recording_id, 'video_annotation_length_mismatch',
                video_frame_count=frame_count, annotation_entries=len(boxes),
            ))
        for frame_index, box in enumerate(boxes, start=1):
            if box is not None:
                issue_type = xywh_issue(box, width, height)
                if issue_type:
                    issues.append(manifest_issue(
                        'drone_detection_dataset2', recording_id, issue_type,
                        original_frame=frame_index, bbox=json.dumps(box),
                    ))
        video_rows.append(row)
    return pd.DataFrame(video_rows), pd.DataFrame(issues)

In [8]:
# Execute this cell after the raw-copy gate reports COMPLETE for both sources.
# It creates only derived CSVs; raw sources remain untouched.
require_complete_raw_data()
MANIFEST_ROOT.mkdir(parents=True, exist_ok=True)

dataset1_manifest, dataset1_issues = inspect_dataset1_train()
dataset2_manifest, dataset2_issues = inspect_dataset2_videos()
issues = pd.concat([dataset1_issues, dataset2_issues], ignore_index=True)

dataset1_manifest.to_csv(MANIFEST_ROOT / 'dataset1_sequences.csv', index=False)
dataset2_manifest.to_csv(MANIFEST_ROOT / 'dataset2_videos.csv', index=False)
issues.to_csv(MANIFEST_ROOT / 'annotation_issues.csv', index=False)

print(f'Dataset 1 sequences: {len(dataset1_manifest):,}')
print(f'Dataset 2 videos: {len(dataset2_manifest):,}')
print('Issue counts:')
display(issues['issue_type'].value_counts(dropna=False).rename_axis('issue_type').to_frame('count'))

Dataset 1 sequences: 223
Dataset 2 videos: 365
Issue counts:


,count
issue_type,
box_exceeds_image_boundary,130
non_positive_box_dimensions,121
unreadable_source_frame,117
video_annotation_length_mismatch,85


## 2.1 Deterministic visual-QA overlays

Review representative valid targets and every discovered issue category before exporting any YOLO labels. These images are inspection artifacts only; the source frames remain unchanged and no overlay is used for model training.

In [9]:
from PIL import ImageDraw, ImageFont

def parse_bbox(value):
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return None
    if isinstance(value, str):
        value = json.loads(value)
    if not isinstance(value, (list, tuple)) or len(value) != 4:
        return None
    return tuple(float(component) for component in value)

def render_overlay(image, bbox, caption, output_path):
    rendered = image.convert('RGB').copy()
    draw = ImageDraw.Draw(rendered)
    if bbox is not None:
        x, y, width, height = bbox
        draw.rectangle((x, y, x + width, y + height), outline='lime', width=3)
    font = ImageFont.load_default()
    bounds = draw.textbbox((0, 0), caption, font=font)
    draw.rectangle((4, 4, bounds[2] + 12, bounds[3] + 12), fill='black')
    draw.text((8, 8), caption, fill='lime', font=font)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    rendered.save(output_path)

def read_dataset2_frame(recording_id, frame_number):
    capture = cv2.VideoCapture(str(DATASET2_ROOT / f'{recording_id}.mp4'))
    try:
        capture.set(cv2.CAP_PROP_POS_FRAMES, frame_number - 1)
        ok, frame = capture.read()
    finally:
        capture.release()
    if not ok:
        raise RuntimeError(f'Unable to decode {recording_id} frame {frame_number}')
    return Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))

In [10]:
# Execute after `write-manifests`. This creates derived QA images and an index only.
dataset1_manifest = pd.read_csv(MANIFEST_ROOT / 'dataset1_sequences.csv')
dataset2_manifest = pd.read_csv(MANIFEST_ROOT / 'dataset2_videos.csv')
issues = pd.read_csv(MANIFEST_ROOT / 'annotation_issues.csv')
qa_records = []

def add_overlay(source_dataset, sequence, frame_number, class_name, bbox, sample_kind, issue_type=None):
    number = len(qa_records) + 1
    filename = f'{number:02d}_{source_dataset}_{sequence}_{frame_number:06d}.png'
    output_path = QA_ROOT / filename
    caption = f'{class_name} | {sample_kind} | frame {frame_number}'
    if issue_type:
        caption += f' | {issue_type}'
    if source_dataset == 'anti_uav_dataset1':
        frame_path = source_frames(DATASET1_ROOT / 'train' / sequence)[frame_number - 1]
        with Image.open(frame_path) as image:
            render_overlay(image, bbox, caption, output_path)
    else:
        render_overlay(read_dataset2_frame(sequence, frame_number), bbox, caption, output_path)
    qa_records.append({
        'source_dataset': source_dataset, 'source_sequence': sequence,
        'original_frame': frame_number, 'class_name': class_name,
        'sample_kind': sample_kind, 'issue_type': issue_type,
        'bbox': json.dumps(bbox) if bbox is not None else None,
        'output_path': output_path.relative_to(REPO_ROOT).as_posix(),
    })

# Three valid Dataset 1 drone examples.
for _, row in dataset1_manifest.sort_values('source_sequence').iloc[:3].iterrows():
    labels = json.loads((REPO_ROOT / row.annotation_path).read_text(encoding='utf-8'))
    for frame_number, (exists_flag, bbox) in enumerate(zip(labels['exist'], labels['gt_rect']), start=1):
        if exists_flag == 1 and xywh_issue(bbox, row.image_width, row.image_height) is None:
            add_overlay('anti_uav_dataset1', row.source_sequence, frame_number, 'Drone', parse_bbox(bbox), 'valid_target')
            break

# Three Dataset 2 examples for each export policy. Bird boxes are visual-QA aids only, never model labels.
for source_class, class_name in (('DRONE', 'Drone positive'), ('BIRD', 'Bird hard negative')):
    for _, row in dataset2_manifest.query('source_class == @source_class').sort_values('source_sequence').head(3).iterrows():
        boxes = load_groundtruth(REPO_ROOT / row.annotation_path)
        for frame_number, bbox in enumerate(boxes, start=1):
            if bbox is not None and xywh_issue(bbox, row.image_width, row.image_height) is None:
                add_overlay('drone_detection_dataset2', row.source_sequence, frame_number, class_name, parse_bbox(bbox), 'valid_target')
                break

# Two examples for each Dataset 1 invalid-box category.
for issue_type in ('box_exceeds_image_boundary', 'non_positive_box_dimensions'):
    for _, row in issues.query("source_dataset == 'anti_uav_dataset1' and issue_type == @issue_type").head(2).iterrows():
        add_overlay('anti_uav_dataset1', row.source_sequence, int(row.original_frame), 'Drone', parse_bbox(row.bbox), 'annotation_issue', issue_type)

# Dataset 2 mismatches: render the last frame in the validated video/label overlap.
for _, issue in issues.query("source_dataset == 'drone_detection_dataset2' and issue_type == 'video_annotation_length_mismatch'").head(3).iterrows():
    row = dataset2_manifest.loc[dataset2_manifest.source_sequence == issue.source_sequence].iloc[0]
    frame_number = min(int(row.video_frame_count), int(row.annotation_entries))
    boxes = load_groundtruth(REPO_ROOT / row.annotation_path)
    bbox = boxes[frame_number - 1] if frame_number <= len(boxes) else None
    class_name = 'Drone' if row.source_class == 'DRONE' else 'Bird'
    add_overlay('drone_detection_dataset2', row.source_sequence, frame_number, class_name, parse_bbox(bbox), 'overlap_boundary', 'video_annotation_length_mismatch')

qa_index = pd.DataFrame(qa_records)
qa_index.to_csv(MANIFEST_ROOT / 'qa_sample_index.csv', index=False)
print(f'Rendered {len(qa_index)} QA overlays to {QA_ROOT}')
display(qa_index)

Rendered 16 QA overlays to D:\Capstone\aai-590-capstone\data\thermal-ir\interim\qa_overlays


,source_dataset,source_sequence,original_frame,class_name,sample_kind,issue_type,bbox,output_path
0,anti_uav_dataset1,01_1751_0250-1750,1,Drone,valid_target,NaN,"[306.0, 270.0, 14.0, 11.0]",data/thermal-ir/interim/qa_overlays/01_anti_ua...
1,anti_uav_dataset1,01_2192_0001-1500,1,Drone,valid_target,NaN,"[268.0, 279.0, 29.0, 22.0]",data/thermal-ir/interim/qa_overlays/02_anti_ua...
2,anti_uav_dataset1,01_4485_1167-2666,4,Drone,valid_target,NaN,"[605.0, 361.0, 27.0, 10.0]",data/thermal-ir/interim/qa_overlays/03_anti_ua...
3,drone_detection_dataset2,IR_DRONE_001,1,Drone positive,valid_target,NaN,"[26.76181411743164, 118.27247619628906, 28.563...",data/thermal-ir/interim/qa_overlays/04_drone_d...
4,drone_detection_dataset2,IR_DRONE_002,1,Drone positive,valid_target,NaN,"[165.0, 113.0, 29.0, 13.0]",data/thermal-ir/interim/qa_overlays/05_drone_d...
5,drone_detection_dataset2,IR_DRONE_003,1,Drone positive,valid_target,NaN,"[177.0, 144.0, 24.0, 13.0]",data/thermal-ir/interim/qa_overlays/06_drone_d...
6,drone_detection_dataset2,IR_BIRD_001,2,Bird hard negative,valid_target,NaN,"[3.0, 153.0, 8.0, 9.0]",data/thermal-ir/interim/qa_overlays/07_drone_d...
7,drone_detection_dataset2,IR_BIRD_002,2,Bird hard negative,valid_target,NaN,"[120.0, 136.0, 9.0, 9.0]",data/thermal-ir/interim/qa_overlays/08_drone_d...
8,drone_detection_dataset2,IR_BIRD_003,2,Bird hard negative,valid_target,NaN,"[158.0, 141.0, 13.0, 14.0]",data/thermal-ir/interim/qa_overlays/09_drone_d...
9,anti_uav_dataset1,01_4485_1167-2666,1,Drone,annotation_issue,box_exceeds_image_boundary,"[633.0, 365.0, 8.0, 8.0]",data/thermal-ir/interim/qa_overlays/10_anti_ua...


# 3. Single-class YOLO export and recording-disjoint split

### 3.1 Export behavior

This stage creates the exact artifacts required by the training guard: the YOLO image/label folders, `metadata/frames.csv`, `metadata/split_integrity_report.json`, and both experiment YAML files. It is deliberately separate from source inspection. Re-running it never alters raw data.

### 3.2 Safety and rerun rule

The export refuses to write into a non-empty output directory. This prevents a partial rerun from mixing files produced under different policies. For recovery after an interrupted run, pass `archive_existing=True`; it renames (does not delete) the previous derived directory before rebuilding. Unreadable source images are skipped and recorded in `metadata/skipped_unreadable_source_frames.csv`.

### 3.3 Temporal sampling and splits

To reduce near-duplicate video frames, this initial baseline exports every fifth eligible frame. Splits are assigned once per source sequence/video, stratified by export policy. Therefore no recording can appear in more than one split.

In [11]:
import random
import shutil
from datetime import datetime

import yaml

EXPORT_FRAME_STRIDE = 5
SPLIT_RATIOS = {'train': 0.70, 'val': 0.15, 'test': 0.15}

def normalised_drone_label(box, width, height):
    x, y, box_width, box_height = (float(value) for value in box)
    return f'0 {(x + box_width / 2) / width:.8f} {(y + box_height / 2) / height:.8f} {box_width / width:.8f} {box_height / height:.8f}\n'

def recording_splits(dataset1_manifest, dataset2_manifest):
    groups = []
    for _, row in dataset1_manifest.iterrows():
        groups.append(('anti_uav_dataset1', row.source_sequence, 'positive_drone'))
    for _, row in dataset2_manifest.iterrows():
        if row.source_class in DATASET2_EXPORT_POLICY:
            groups.append(('drone_detection_dataset2', row.source_sequence, DATASET2_EXPORT_POLICY[row.source_class]))
    assignments = {}
    for policy in sorted({policy for _, _, policy in groups}):
        members = [(source, sequence) for source, sequence, item_policy in groups if item_policy == policy]
        random.Random(f'{RANDOM_SEED}:{policy}').shuffle(members)
        train_end = round(len(members) * SPLIT_RATIOS['train'])
        val_end = train_end + round(len(members) * SPLIT_RATIOS['val'])
        for index, member in enumerate(members):
            assignments[member] = 'train' if index < train_end else 'val' if index < val_end else 'test'
    return assignments

def write_yolo_yaml(path, split_sources):
    yaml.safe_dump({'path': str(YOLO_ROOT), **split_sources, 'names': {0: 'drone'}}, path.open('w', encoding='utf-8'), sort_keys=False)

def build_single_class_yolo_dataset(frame_stride=EXPORT_FRAME_STRIDE, archive_existing=False):
    """Export a new one-class dataset; raw sources are read only and prior derived output can be archived."""
    require_complete_raw_data()
    if frame_stride < 1:
        raise ValueError('frame_stride must be at least 1.')
    if YOLO_ROOT.exists() and any(YOLO_ROOT.iterdir()):
        if not archive_existing:
            raise RuntimeError(f'{YOLO_ROOT} is not empty. Re-run with archive_existing=True to preserve it and rebuild.')
        archive_path = YOLO_ROOT.with_name(f'{YOLO_ROOT.name}_partial_{datetime.now():%Y%m%d_%H%M%S}')
        YOLO_ROOT.rename(archive_path)
        print(f'Archived previous derived export to {archive_path}')
    dataset1_manifest = pd.read_csv(MANIFEST_ROOT / 'dataset1_sequences.csv')
    dataset2_manifest = pd.read_csv(MANIFEST_ROOT / 'dataset2_videos.csv')
    issues = pd.read_csv(MANIFEST_ROOT / 'annotation_issues.csv')
    invalid_dataset1 = {(row.source_sequence, int(row.original_frame)) for _, row in issues.query("source_dataset == 'anti_uav_dataset1' and issue_type in ['box_exceeds_image_boundary', 'non_positive_box_dimensions', 'unreadable_source_frame']").iterrows()}
    assignments = recording_splits(dataset1_manifest, dataset2_manifest)
    for split in SPLIT_RATIOS:
        (YOLO_ROOT / 'images' / split).mkdir(parents=True, exist_ok=True)
        (YOLO_ROOT / 'labels' / split).mkdir(parents=True, exist_ok=True)
    records, skipped_unreadable = [], []
    def emit(image, split, source_dataset, sequence, frame_number, source_class, export_policy, label_text, action):
        stem = f'{source_dataset}_{sequence}_{frame_number:06d}'
        image_path = YOLO_ROOT / 'images' / split / f'{stem}.jpg'
        label_path = YOLO_ROOT / 'labels' / split / f'{stem}.txt'
        image.convert('RGB').save(image_path, quality=95)
        label_path.write_text(label_text, encoding='utf-8')
        records.append({'source_dataset': source_dataset, 'source_sequence': sequence, 'original_frame': frame_number, 'source_class': source_class, 'export_policy': export_policy, 'label_status': 'drone_positive' if label_text else 'no_drone', 'cleaning_action': action, 'split': split, 'image_path': image_path.relative_to(REPO_ROOT).as_posix(), 'label_path': label_path.relative_to(REPO_ROOT).as_posix()})
    for _, row in dataset1_manifest.iterrows():
        sequence, split = row.source_sequence, assignments[('anti_uav_dataset1', row.source_sequence)]
        labels = json.loads((REPO_ROOT / row.annotation_path).read_text(encoding='utf-8'))
        for frame_number, (exists_flag, box, frame_path) in enumerate(zip(labels['exist'], labels['gt_rect'], source_frames(DATASET1_ROOT / 'train' / sequence)), start=1):
            if frame_number % frame_stride or exists_flag != 1 or (sequence, frame_number) in invalid_dataset1:
                continue
            try:
                with Image.open(frame_path) as image:
                    image.load()
                    emit(image, split, 'anti_uav_dataset1', sequence, frame_number, 'DRONE', 'positive_drone', normalised_drone_label(box, image.width, image.height), 'exported_valid_drone')
            except (UnidentifiedImageError, OSError) as error:
                skipped_unreadable.append({'source_dataset': 'anti_uav_dataset1', 'source_sequence': sequence, 'original_frame': frame_number, 'source_path': frame_path.relative_to(REPO_ROOT).as_posix(), 'reason': str(error)})
    for _, row in dataset2_manifest.iterrows():
        if row.source_class not in DATASET2_EXPORT_POLICY:
            continue
        sequence, policy = row.source_sequence, DATASET2_EXPORT_POLICY[row.source_class]
        split = assignments[('drone_detection_dataset2', sequence)]
        boxes = load_groundtruth(REPO_ROOT / row.annotation_path)
        limit = min(int(row.video_frame_count), len(boxes))
        capture = cv2.VideoCapture(str(REPO_ROOT / row.video_path))
        try:
            for frame_number in range(1, limit + 1):
                ok, frame = capture.read()
                if not ok:
                    raise RuntimeError(f'Unable to decode {sequence} frame {frame_number}')
                if frame_number % frame_stride:
                    continue
                box = boxes[frame_number - 1]
                if policy == 'positive_drone' and (box is None or xywh_issue(box, int(row.image_width), int(row.image_height)) is not None):
                    continue
                image = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
                label_text = normalised_drone_label(box, image.width, image.height) if policy == 'positive_drone' else ''
                emit(image, split, 'drone_detection_dataset2', sequence, frame_number, row.source_class, policy, label_text, 'exported_valid_drone' if label_text else 'exported_bird_hard_negative')
        finally:
            capture.release()
    frames = pd.DataFrame(records)
    if frames.empty or not (frames.label_status == 'drone_positive').any() or not (frames.export_policy == 'negative_bird').any():
        raise RuntimeError('Export did not produce both drone-positive and bird-negative examples.')
    metadata_root = YOLO_ROOT / 'metadata'; metadata_root.mkdir(parents=True, exist_ok=True)
    frames.to_csv(metadata_root / 'frames.csv', index=False)
    pd.DataFrame(skipped_unreadable).to_csv(metadata_root / 'skipped_unreadable_source_frames.csv', index=False)
    group_counts = frames.groupby(['source_dataset', 'source_sequence']).split.nunique()
    if (group_counts != 1).any():
        raise RuntimeError('Split leakage detected: a recording appears in multiple splits.')
    split_sources = {split: f'images/{split}' for split in SPLIT_RATIOS}
    write_yolo_yaml(YOLO_ROOT / 'experiment_b_combined.yaml', split_sources)
    split_root = YOLO_ROOT / 'splits'; split_root.mkdir(exist_ok=True)
    dataset2_only = frames.source_dataset.eq('drone_detection_dataset2')
    for split in SPLIT_RATIOS:
        paths = [str((REPO_ROOT / path).resolve()) for path in frames.loc[dataset2_only & frames.split.eq(split), 'image_path']]
        (split_root / f'dataset2_only_{split}.txt').write_text('\n'.join(paths) + '\n', encoding='utf-8')
    write_yolo_yaml(YOLO_ROOT / 'experiment_a_dataset2_only.yaml', {split: f'splits/dataset2_only_{split}.txt' for split in SPLIT_RATIOS})
    report = {'frame_stride': frame_stride, 'split_ratios': SPLIT_RATIOS, 'frame_counts': frames.groupby(['split', 'export_policy']).size().reset_index(name='frames').to_dict('records'), 'recording_counts': frames.drop_duplicates(['source_dataset', 'source_sequence']).groupby(['split', 'export_policy']).size().reset_index(name='recordings').to_dict('records'), 'unreadable_source_frames_skipped': len(skipped_unreadable), 'recording_disjoint': True}
    (metadata_root / 'split_integrity_report.json').write_text(json.dumps(report, indent=2, default=str), encoding='utf-8')
    return frames


In [12]:
# Reuse a completed derived export; archive and rebuild only an incomplete export.
frames_path = YOLO_ROOT / 'metadata' / 'frames.csv'
integrity_path = YOLO_ROOT / 'metadata' / 'split_integrity_report.json'
if frames_path.is_file() and integrity_path.is_file():
    frames = pd.read_csv(frames_path)
    print(f'Reusing completed YOLO export: {YOLO_ROOT}')
else:
    recovery_needed = YOLO_ROOT.exists() and any(YOLO_ROOT.iterdir())
    frames = build_single_class_yolo_dataset(archive_existing=recovery_needed)
display(frames.groupby(['split', 'export_policy', 'label_status']).size().rename('frames').to_frame())
print(f'Prepared YOLO dataset: {YOLO_ROOT}')

Reusing completed YOLO export: D:\Capstone\aai-590-capstone\data\thermal-ir\processed\yolo_v1


frames
split export_policy  label_status          
test  negative_bird  no_drone           753
      positive_drone drone_positive    7217
train negative_bird  no_drone          3369
      positive_drone drone_positive   39945
val   negative_bird  no_drone           741
      positive_drone drone_positive    8174

Prepared YOLO dataset: D:\Capstone\aai-590-capstone\data\thermal-ir\processed\yolo_v1


# 4. Model creation and training (approval-gated)

The detector will use a lightweight pretrained Ultralytics YOLO detection model with one class, `drone`. It runs on PyTorch and CUDA. Use **one GPU** (`device=0`) on Windows. The model section remains blocked until the single-class export, deduplication, and recording-disjoint split stages produce a validated `data.yaml`, frame metadata, and split-integrity report.

### 4.1 Training objective

A detection is interpreted as **drone present** when at least one class-0 prediction meets the selected confidence threshold. A bird frame has an empty YOLO label file, so a prediction on it is a false positive for the deployed question—not a bird-class error.

### 4.2 Model creation

The baseline instantiates a pretrained YOLO nano detector with a one-class dataset configuration. Pretraining supplies general visual features; fine-tuning adapts the detection head to thermal drone localization. The code below creates a fresh model instance for each experiment, so runs do not share optimizer state or learned weights.

### 4.3 Training gate and experiment discipline

Training may begin only after raw-data completeness, annotation cleanup, export validation, duplicate checks, and recording-disjoint split validation succeed. Select models and confidence thresholds using validation data only; execute held-out test evaluation once for the selected checkpoint.

The baseline model is `yolo26n.pt`: the nano variant limits initial compute cost while pretrained weights provide a practical transfer-learning baseline. Its suitability must be reported alongside limitations for small thermal targets; it is not an operational-system validation.

Install the CUDA-enabled PyTorch build selected for the local driver from [PyTorch's official selector](https://pytorch.org/get-started/locally/), then install Ultralytics:

```powershell
# Run from the repository root after activating .venv.
# Use the exact CUDA wheel command supplied by pytorch.org for this system.
uv pip install torch torchvision --index-url <official-pytorch-cuda-wheel-index>
uv pip install ultralytics
```

No Docker configuration is required.

### 4.4 Mosaic augmentation

Mosaic is a training-only transformation. Instead of showing the model one source image, YOLO can place four source images into one composite training image. The raw source files, the exported YOLO images on disk, and normal inference images are not changed.

```text
+----------------+----------------+
| source image A | source image B |
+----------------+----------------+
| source image C | source image D |
+----------------+----------------+
```

Every source bounding box is resized and moved with its image. If two of the four source images contain drones, the composite correctly contains two class-0 (`drone`) boxes. The blue `0` labels visible in Ultralytics `train_batch#.jpg` files are ground-truth class IDs, not confidence scores.

The seams, gray padding, and unrelated scenes inside one displayed training tile are expected mosaic behavior. The outer `train_batch` display is a 4×4 grid of training samples; an individual sample within that grid can itself be a four-image mosaic.

This baseline uses `mosaic: 0.5`, so roughly half of training samples are mosaics. `close_mosaic: 10` disables mosaic for the final ten epochs, allowing the model to finish learning from normal, unmosaicked images. Mosaic should be changed only as a controlled augmentation experiment, not because the default preview mosaics are visually cluttered.

The overlapping white text in the default preview comes from Ultralytics drawing long filenames above each tile. It is a plotting artifact and does not affect training or labels. Report-quality QA will instead use curated normal-image panels with short captions, ground-truth boxes, and predictions at the selected decision threshold.

In [13]:
# Execute immediately before training; it fails fast if CUDA PyTorch is unavailable.
import torch
from ultralytics import YOLO

if not torch.cuda.is_available():
    raise RuntimeError(
        'CUDA is unavailable. Install the CUDA-enabled PyTorch wheel selected from pytorch.org, '
        'then restart the notebook kernel.'
    )
print(f'PyTorch: {torch.__version__}')
print(f'CUDA runtime: {torch.version.cuda}')
print(f'Device 0: {torch.cuda.get_device_name(0)}')
print(f'Available memory (GiB): {torch.cuda.mem_get_info(0)[0] / 2**30:.1f}')

PyTorch: 2.14.1+cu130
CUDA runtime: 13.0
Device 0: NVIDIA GeForce RTX 5080
Available memory (GiB): 14.6


In [14]:
# Fixed initial configuration for a controlled single-class Experiment A/B comparison.
# Changes after this point require a documented experiment revision.
MODEL_CONFIG = {
    'model': 'yolo26n.pt',
    'imgsz': 640,
    'epochs': 100,
    'batch': 32,
    'device': 0,
    'optimizer': 'AdamW',
    'lr0': 1e-3,
    'weight_decay': 5e-4,
    'patience': 20,
    'seed': RANDOM_SEED,
    'deterministic': True,
    'workers': 8,
    'pretrained': True,
    'cache': False,
    # Thermal-appropriate geometric augmentation; disable hue/saturation/value color changes.
    'hsv_h': 0.0,
    'hsv_s': 0.0,
    'hsv_v': 0.0,
    'fliplr': 0.5,
    'flipud': 0.0,
    'degrees': 0.0,
    'translate': 0.10,
    'scale': 0.25,
    'mosaic': 0.5,
    'close_mosaic': 10,
}

EXPERIMENTS = {
    'experiment_a_dataset2_only': YOLO_ROOT / 'experiment_a_dataset2_only.yaml',
    'experiment_b_combined': YOLO_ROOT / 'experiment_b_combined.yaml',
}
RUN_NAMES = {
    'experiment_a_dataset2_only': 'experiment_a_dataset2_only_batch32',
    'experiment_b_combined': 'experiment_b_combined_batch32',
}
TRAINING_RUN_ROOT = RESULTS_ROOT / 'runs'
TEST_IOU_THRESHOLD = 0.50
# Keep low-confidence predictions for threshold analysis; choose an operating threshold on validation only.
TEST_CONFIDENCE_THRESHOLD = 0.001
DECISION_CONFIDENCE_THRESHOLDS = (0.05, 0.10, 0.25, 0.50, 0.75)
MIN_VALIDATION_RECALL = 0.90

assert CLASS_NAMES == {0: 'drone'}

def create_baseline_model() -> YOLO:
    """Create an independent pretrained baseline for one single-class experiment."""
    return YOLO(MODEL_CONFIG['model'])

MODEL_CONFIG

{'model': 'yolo26n.pt',
 'imgsz': 640,
 'epochs': 100,
 'batch': 32,
 'device': 0,
 'optimizer': 'AdamW',
 'lr0': 0.001,
 'weight_decay': 0.0005,
 'patience': 20,
 'seed': 590,
 'deterministic': True,
 'workers': 8,
 'pretrained': True,
 'cache': False,
 'hsv_h': 0.0,
 'hsv_s': 0.0,
 'hsv_v': 0.0,
 'fliplr': 0.5,
 'flipud': 0.0,
 'degrees': 0.0,
 'translate': 0.1,
 'scale': 0.25,
 'mosaic': 0.5,
 'close_mosaic': 10}

In [15]:
def require_prepared_yolo_dataset(experiment_yaml: Path) -> None:
    required = [
        experiment_yaml,
        YOLO_ROOT / 'metadata' / 'frames.csv',
        YOLO_ROOT / 'metadata' / 'split_integrity_report.json',
    ]
    missing = [str(path.relative_to(REPO_ROOT)) for path in required if not path.is_file()]
    if missing:
        raise RuntimeError(
            'Training is blocked until the validated exported dataset exists. Missing: ' + ', '.join(missing)
        )

def train_experiment(experiment_name: str):
    """Train one controlled experiment; never use the held-out test split for model selection."""
    if experiment_name not in EXPERIMENTS:
        raise KeyError(f'Unknown experiment: {experiment_name}')
    data_yaml = EXPERIMENTS[experiment_name]
    require_prepared_yolo_dataset(data_yaml)
    run_directory = TRAINING_RUN_ROOT / RUN_NAMES[experiment_name]
    model = create_baseline_model()
    model.train(
        data=str(data_yaml),
        project=str(run_directory.parent),
        name=run_directory.name,
        exist_ok=False,
        **{key: value for key, value in MODEL_CONFIG.items() if key != 'model'},
    )
    return Path(model.trainer.save_dir)


In [16]:
ACTIVE_EXPERIMENT = 'experiment_a_dataset2_only'
training_run_dir = train_experiment(ACTIVE_EXPERIMENT)
BEST_WEIGHTS = training_run_dir / 'weights' / 'best.pt'
if not BEST_WEIGHTS.is_file():
    raise RuntimeError(f'Expected best checkpoint was not written: {BEST_WEIGHTS}')
print(f'Completed training run: {training_run_dir}')
print(f'Best checkpoint: {BEST_WEIGHTS}')

Ultralytics 8.4.198  Python-3.11.7 torch-2.14.1+cu130 CUDA:0 (NVIDIA GeForce RTX 5080, 16276MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=D:\Capstone\aai-590-capstone\data\thermal-ir\processed\yolo_v1\experiment_a_dataset2_only.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.0, hsv_s=0.0, hsv_v=0.0, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26n.pt, momentum=0.937, mosaic=0.5, mul

# 5. Held-out test evaluation

After validation-based model selection is finished, evaluate the selected checkpoint once on the isolated test split. Ultralytics validation supplies precision, recall, mAP@0.5, mAP@0.5:0.95, curves, and a detection confusion matrix. The custom exporter below supplements those artifacts with an auditable row-level test-predictions CSV.

### 5.1 Object-level and decision-level measures

A prediction is matched greedily to the highest-IoU unused ground-truth drone at IoU ≥ 0.50. Rows are labeled `true_positive`, `false_negative`, or `false_positive`. The test report must separately summarize the binary decision **drone present / no drone present** at every candidate confidence threshold. Bird-only frames are the primary no-drone cohort: any retained prediction on them is a directly interpretable false positive.

In [17]:
import numpy as np

def iou_xyxy(first: np.ndarray, second: np.ndarray) -> float:
    left, top = np.maximum(first[:2], second[:2])
    right, bottom = np.minimum(first[2:], second[2:])
    intersection = max(0.0, right - left) * max(0.0, bottom - top)
    first_area = max(0.0, first[2] - first[0]) * max(0.0, first[3] - first[1])
    second_area = max(0.0, second[2] - second[0]) * max(0.0, second[3] - second[1])
    union = first_area + second_area - intersection
    return intersection / union if union else 0.0

def yolo_label_to_xyxy(parts: list[float], width: int, height: int) -> tuple[int, np.ndarray]:
    class_id, x_center, y_center, box_width, box_height = parts
    x_center, box_width = x_center * width, box_width * width
    y_center, box_height = y_center * height, box_height * height
    return int(class_id), np.array([x_center - box_width / 2, y_center - box_height / 2, x_center + box_width / 2, y_center + box_height / 2])

def read_ground_truth(label_path: Path, width: int, height: int):
    if not label_path.is_file() or not label_path.read_text().strip():
        return []
    return [yolo_label_to_xyxy([float(value) for value in line.split()], width, height) for line in label_path.read_text().splitlines()]

def experiment_split_rows(metadata: pd.DataFrame, experiment_yaml: Path, split: str) -> pd.DataFrame:
    """Select manifest rows actually referenced by an experiment's YAML split."""
    config = yaml.safe_load(experiment_yaml.read_text(encoding='utf-8'))
    split_source = Path(config[split])
    root = Path(config.get('path', YOLO_ROOT))
    split_source = split_source if split_source.is_absolute() else root / split_source
    rows = metadata.loc[metadata.split.eq(split)].copy()
    if split_source.suffix.lower() != '.txt':
        return rows
    listed_images = {Path(value).resolve() for value in split_source.read_text(encoding='utf-8').splitlines() if value.strip()}
    selected = rows.loc[rows.image_path.map(lambda value: (REPO_ROOT / value).resolve() in listed_images)].copy()
    if selected.empty:
        raise RuntimeError(f'No manifest rows matched {split_source}.')
    return selected

def threshold_summary(decisions: pd.DataFrame, threshold: float) -> tuple[pd.DataFrame, dict]:
    if not 0.0 < threshold <= 1.0:
        raise ValueError('decision threshold must be in (0, 1].')
    evaluated = decisions.copy()
    evaluated['predicted_drone_present'] = evaluated.max_confidence.ge(threshold)
    evaluated['binary_outcome'] = np.select(
        [evaluated.ground_truth_drone_present & evaluated.predicted_drone_present, ~evaluated.ground_truth_drone_present & evaluated.predicted_drone_present, evaluated.ground_truth_drone_present & ~evaluated.predicted_drone_present],
        ['true_positive', 'false_positive', 'false_negative'], default='true_negative',
    )
    bird_rows = evaluated.export_policy.eq('negative_bird')
    positives = evaluated.ground_truth_drone_present
    true_positive = int((evaluated.binary_outcome == 'true_positive').sum())
    false_positive = int((evaluated.binary_outcome == 'false_positive').sum())
    false_negative = int((evaluated.binary_outcome == 'false_negative').sum())
    true_negative = int((evaluated.binary_outcome == 'true_negative').sum())
    precision = true_positive / (true_positive + false_positive) if true_positive + false_positive else 0.0
    recall = true_positive / (true_positive + false_negative) if true_positive + false_negative else 0.0
    metrics = {
        'decision_threshold': threshold,
        'true_positive': true_positive,
        'false_positive': false_positive,
        'false_negative': false_negative,
        'true_negative': true_negative,
        'precision': precision,
        'recall': recall,
        'f1': 2 * precision * recall / (precision + recall) if precision + recall else 0.0,
        'overall_false_positive_rate': false_positive / (false_positive + true_negative) if false_positive + true_negative else 0.0,
        'bird_false_positive_rate': float(evaluated.loc[bird_rows, 'predicted_drone_present'].mean()),
        'drone_false_negative_rate': float((~evaluated.loc[positives, 'predicted_drone_present']).mean()),
    }
    return evaluated, metrics

def export_heldout_predictions(best_weights: Path, experiment_yaml: Path, decision_threshold: float) -> tuple[pd.DataFrame, pd.DataFrame, dict]:
    """Run final inference once and write row-level matched/unmatched test outcomes."""
    require_prepared_yolo_dataset(experiment_yaml)
    metadata = pd.read_csv(YOLO_ROOT / 'metadata' / 'frames.csv')
    test_rows = experiment_split_rows(metadata, experiment_yaml, 'test')
    if test_rows.empty:
        raise RuntimeError('No held-out test rows found in frames.csv.')
    model = YOLO(str(best_weights))
    results = model.val(data=str(experiment_yaml), split='test', plots=True, conf=TEST_CONFIDENCE_THRESHOLD)
    experiment_name = best_weights.parent.parent.name
    evaluation_root = RESULTS_ROOT / 'evaluation' / experiment_name
    evaluation_root.mkdir(parents=True, exist_ok=True)
    metrics_path = evaluation_root / 'test_detection_metrics.json'
    metrics_path.parent.mkdir(parents=True, exist_ok=True)
    metrics_path.write_text(json.dumps(results.results_dict, indent=2), encoding='utf-8')
    exported, image_decisions = [], []
    for _, row in test_rows.iterrows():
        image_path = REPO_ROOT / row.image_path
        label_path = REPO_ROOT / row.label_path
        with Image.open(image_path) as image:
            width, height = image.size
        ground_truth = read_ground_truth(label_path, width, height)
        prediction = model.predict(str(image_path), conf=TEST_CONFIDENCE_THRESHOLD, verbose=False)[0]
        predicted = [
            (int(class_id), float(confidence), box.cpu().numpy())
            for class_id, confidence, box in zip(prediction.boxes.cls, prediction.boxes.conf, prediction.boxes.xyxy)
        ]
        image_decisions.append({
            **row.to_dict(),
            'ground_truth_drone_present': bool(ground_truth),
            'predicted_drone_present': bool(predicted),
            'max_confidence': max((confidence for _, confidence, _ in predicted), default=0.0),
            **{f'drone_present_at_{threshold:.2f}': any(confidence >= threshold for _, confidence, _ in predicted) for threshold in DECISION_CONFIDENCE_THRESHOLDS},
        })
        pairs = sorted(
            ((iou_xyxy(gt_box, pred_box), gt_index, pred_index) for gt_index, (_, gt_box) in enumerate(ground_truth) for pred_index, (_, _, pred_box) in enumerate(predicted)),
            reverse=True,
        )
        matched_gt, matched_pred = set(), set()
        for iou, gt_index, pred_index in pairs:
            if iou < TEST_IOU_THRESHOLD or gt_index in matched_gt or pred_index in matched_pred:
                continue
            matched_gt.add(gt_index); matched_pred.add(pred_index)
            gt_class, gt_box = ground_truth[gt_index]
            pred_class, confidence, pred_box = predicted[pred_index]
            exported.append({**row.to_dict(), 'ground_truth_class_id': gt_class, 'predicted_class_id': pred_class, 'confidence': confidence, 'iou': iou, 'outcome': 'true_positive', 'ground_truth_xyxy': gt_box.tolist(), 'predicted_xyxy': pred_box.tolist()})
        for gt_index, (gt_class, gt_box) in enumerate(ground_truth):
            if gt_index not in matched_gt:
                exported.append({**row.to_dict(), 'ground_truth_class_id': gt_class, 'predicted_class_id': None, 'confidence': None, 'iou': 0.0, 'outcome': 'false_negative', 'ground_truth_xyxy': gt_box.tolist(), 'predicted_xyxy': None})
        for pred_index, (pred_class, confidence, pred_box) in enumerate(predicted):
            if pred_index not in matched_pred:
                exported.append({**row.to_dict(), 'ground_truth_class_id': None, 'predicted_class_id': pred_class, 'confidence': confidence, 'iou': 0.0, 'outcome': 'false_positive', 'ground_truth_xyxy': None, 'predicted_xyxy': pred_box.tolist()})
    report = pd.DataFrame(exported)
    output = evaluation_root / 'test_predictions.csv'
    output.parent.mkdir(parents=True, exist_ok=True)
    report.to_csv(output, index=False)
    image_decision_output = evaluation_root / 'test_image_decisions.csv'
    decisions, decision_metrics = threshold_summary(pd.DataFrame(image_decisions), decision_threshold)
    decisions.to_csv(image_decision_output, index=False)
    (evaluation_root / 'test_decision_metrics.json').write_text(json.dumps(decision_metrics, indent=2), encoding='utf-8')
    return report, decisions, decision_metrics

def evaluate_validation_thresholds(best_weights: Path, experiment_yaml: Path, thresholds=DECISION_CONFIDENCE_THRESHOLDS) -> pd.DataFrame:
    """Score candidate image-level decision thresholds on validation data only."""
    require_prepared_yolo_dataset(experiment_yaml)
    metadata = pd.read_csv(YOLO_ROOT / 'metadata' / 'frames.csv')
    validation_rows = experiment_split_rows(metadata, experiment_yaml, 'val')
    model = YOLO(str(best_weights))
    scores = []
    for _, row in validation_rows.iterrows():
        prediction = model.predict(str(REPO_ROOT / row.image_path), conf=TEST_CONFIDENCE_THRESHOLD, verbose=False)[0]
        confidences = [float(value) for value in prediction.boxes.conf]
        scores.append({**row.to_dict(), 'ground_truth_drone_present': row.label_status == 'drone_positive', 'max_confidence': max(confidences, default=0.0)})
    decisions = pd.DataFrame(scores)
    summaries = []
    for threshold in thresholds:
        _, summary = threshold_summary(decisions, float(threshold))
        summaries.append(summary)
    report = pd.DataFrame(summaries).sort_values('decision_threshold')
    evaluation_root = RESULTS_ROOT / 'evaluation' / best_weights.parent.parent.name
    evaluation_root.mkdir(parents=True, exist_ok=True)
    report.to_csv(evaluation_root / 'validation_threshold_report.csv', index=False)
    return report

def select_validation_threshold(report: pd.DataFrame, minimum_recall=MIN_VALIDATION_RECALL) -> float:
    """Minimize bird false positives while retaining the configured minimum validation recall."""
    eligible = report.loc[report.recall.ge(minimum_recall)].copy()
    candidates = eligible if not eligible.empty else report.copy()
    ranked = candidates.sort_values(['bird_false_positive_rate', 'f1', 'decision_threshold'], ascending=[True, False, False])
    return float(ranked.iloc[0].decision_threshold)

validation_thresholds = evaluate_validation_thresholds(BEST_WEIGHTS, EXPERIMENTS[ACTIVE_EXPERIMENT])
DECISION_THRESHOLD = select_validation_threshold(validation_thresholds)
display(validation_thresholds)
print(f'Selected validation threshold: {DECISION_THRESHOLD:.2f}')

test_report, test_decisions, test_summary = export_heldout_predictions(
    BEST_WEIGHTS,
    EXPERIMENTS[ACTIVE_EXPERIMENT],
    decision_threshold=DECISION_THRESHOLD,
)
display(pd.Series(test_summary))

,decision_threshold,true_positive,false_positive,false_negative,true_negative,precision,recall,f1,overall_false_positive_rate,bird_false_positive_rate,drone_false_negative_rate
0,0.05,1430,2,0,739,0.998603,1.000000,0.999301,0.002699,0.002699,0.000000
1,0.10,1429,2,1,739,0.998602,0.999301,0.998951,0.002699,0.002699,0.000699
2,0.25,1403,1,27,740,0.999288,0.981119,0.990120,0.001350,0.001350,0.018881
3,0.50,762,0,668,741,1.000000,0.532867,0.695255,0.000000,0.000000,0.467133
4,0.75,0,0,1430,741,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000


Selected validation threshold: 0.25
Ultralytics 8.4.198  Python-3.11.7 torch-2.14.1+cu130 CUDA:0 (NVIDIA GeForce RTX 5080, 16276MiB)
YOLO26n summary (fused): 121 layers, 2,389,735 parameters, 0 gradients, 5.4 GFLOPs
WARNING val: Slow image access detected (ping: 0.0±0.0 ms, read: 6.3±3.0 MB/s, size: 21.3 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips
val: Scanning D:\Capstone\aai-590-capstone\data\thermal-ir\processed\yolo_v1\labels\test... 2631 images, 753 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 2631/2631 1.4Kit/s 1.9s0.0s
val: New cache created: D:\Capstone\aai-590-capstone\data\thermal-ir\processed\yolo_v1\labels\test.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 165/165 8.2it/s 20.0s<0.1s
                   all       2631       1878      0.875      0.875       0.88      0.395
Speed: 0.6ms preprocess, 0.8ms inference, 0.0

decision_threshold                0.250000
true_positive                  1830.000000
false_positive                    5.000000
false_negative                   48.000000
true_negative                   748.000000
precision                         0.997275
recall                            0.974441
f1                                0.985726
overall_false_positive_rate       0.006640
bird_false_positive_rate          0.006640
drone_false_negative_rate         0.025559
dtype: float64

# 6. Single-class export and split contract

## 6.1 Label policy

The exported YOLO dataset has exactly one class: `0 drone`. A Dataset 1 frame contributes a positive label only when `exist[i] == 1` and its native `xywh` box is valid. A Dataset 2 `DRONE` frame with a valid decoded box contributes the same positive label. A Dataset 2 `BIRD` frame is retained as a **no-drone hard negative**, even when its source annotation contains a bird box; its YOLO label file is empty.

Do not convert a malformed or out-of-bounds Dataset 1 drone annotation into an empty label. The target was marked present, so such frames are excluded pending review rather than being mislabeled as no-drone.

## 6.2 Cleanup and provenance

When raw data are available locally, the export must:

1. Create immutable per-sequence/video manifests and issue logs.
2. Validate Dataset 1 `exist`/`gt_rect` alignment and Dataset 2 decoded `.mat`/video alignment.
3. Render a reviewed sample of overlays. Overlay images are QA artifacts only; no boxes are drawn into model inputs.
4. Export validated thermal frames with separate YOLO labels and provenance metadata. Bird label files remain empty.
5. Apply deterministic temporal subsampling, duplicate checks, and recording-disjoint splits.
6. Stop for approval before any YOLO training.


## 6.3 Mismatch and boundary rules

For every Dataset 2 video, process only `min(video_frame_count, annotation_entries)` frames and retain both source counts in the manifest. Never pad missing annotations or manufacture tail-frame labels. For the 130 Dataset 1 boxes beyond image boundaries, exclude the affected positive frame by default; clipping is allowed only after a reviewed, documented policy is adopted. For the 121 non-positive boxes, exclude the affected positive frame.

## 6.4 Split integrity

Assign `train`, `validation`, and `test` at the source-sequence/video level before frame extraction. Do not let adjacent frames from a recording cross splits. Store source dataset, recording ID, original frame number, source class, export policy, label status, cleaning action, and split in `metadata/frames.csv`; validate that every recording appears in exactly one split.

Excluded Dataset 2 airplane and helicopter recordings will not be treated as unlabeled background. Dataset 1 `track1_test` and `track2_test` remain inference-only.

# 7. Decision-focused evaluation and outputs

## 7.1 Decision rule and threshold selection

The one-class detector's boxes are converted into an image-level decision using the confidence threshold selected on validation data only. That threshold is then locked before the single held-out test evaluation. Report both object-detection quality and the thresholded image-level decision.

## 7.2 Required held-out reports

After the dataset passes validation and training is approved, this notebook will add a lightweight maintained YOLO configuration, exact run commands, and fixed-seed Experiment A/B evaluation. The held-out test set will stay fixed and Dataset 2-only. Alongside mAP and recall, report overall false-positive rate, bird-only false-positive rate, false-negative rate on drone-present images, precision, recall, F1, and a binary confusion matrix.

Expected result artifacts:

- `results/thermal-ir/metrics/test_metrics.json`
- `results/thermal-ir/figures/confusion_matrix.png`
- `results/thermal-ir/test_predictions.csv` — one row per held-out ground-truth object and unmatched prediction, including image ID, source, confidence, IoU/match status, and error type.
- `results/thermal-ir/test_image_decisions.csv` — one row per held-out image, including source policy, maximum confidence, thresholded drone-present decision, and binary outcome.

No metrics, confusion matrix, or predictions will be fabricated before a trained model is evaluated on the isolated test split.